# Quick start: inspect five micro-watersheds

This smallest CoRE Stack notebook confirms that browser Python works, fetches five published features, shows their attributes, and adds them to the adjacent GeoLibre map. Run the four code cells in order with **Shift+Enter**. No package installation is needed.

## 1. Confirm the browser kernel

In [ ]:
import json
SCOPE = json.loads("{\"state\":\"Jharkhand\",\"district\":\"Dumka\",\"tehsil\":\"Masalia\",\"bounds\":[86.89,23.94,87.24,24.28]}")
print(f"Python is ready for {SCOPE['tehsil']}, {SCOPE['district']}. No packages were installed.")

## 2. Fetch five features

This uses only Pyodide's browser HTTP helper and a bounded GeoServer WFS request.

In [ ]:
import re
from pyodide.http import pyfetch
district, tehsil = [re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_") for value in (SCOPE["district"], SCOPE["tehsil"])]
layer_name = f"deltaG_well_depth_{district}_{tehsil}"
url = f"https://geoserver.core-stack.org:8443/geoserver/mws_layers/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=mws_layers:{layer_name}&outputFormat=application/json&srsName=EPSG:4326&maxFeatures=5"
response = await pyfetch(url)
if not response.ok: raise RuntimeError(f"GeoServer returned HTTP {response.status}.")
data = await response.json()
print(f"Loaded {len(data['features'])} micro-watersheds from {SCOPE['tehsil']}.")

## 3. Inspect the attributes

Expand a record to see annual groundwater values, area, net-change summaries, and its MWS identifier.

In [ ]:
from IPython.display import JSON, display
attributes = [feature.get("properties", {}) for feature in data["features"]]
display(JSON(attributes, expanded=False))

## 4. Add the same five features to GeoLibre

The new layer is temporary and does not change the published CoRE Stack data.

In [ ]:
import geolibre
m = geolibre.connect()
layer_id = m.add_geojson(data, name="Notebook · five MWS preview", fillColor="#60a5fa", strokeColor="#1e3a8a", fillOpacity=0.35)
print(f"Added temporary GeoLibre layer: {layer_id}")